In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModel
from scipy.stats import spearmanr
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

f:\Programs\PYTHON_310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
f:\Programs\PYTHON_310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory: 6.0 GB


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "bert-base-uncased"

DATA_DIR = Path("..") / "data"
STS_DIR = DATA_DIR / "STS"

NLI_MODEL_DIR = Path("outputs") / "sbert_nli_base"
NLI_MODEL_PATH = (
    NLI_MODEL_DIR
    / "model"
    / "sbert_nli_base.pt"
)

NLI_TOKENIZER_PATH = (
    NLI_MODEL_DIR
    / "tokenizer"
)

OUTPUT_DIR = Path("outputs") / "sbert_nli_stsb"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Paper-related configuration
# ------------------------------------------------------------

EPOCHS = 4
LEARNING_RATE = 2e-5

# The paper's Table 2 experiments use 4 epochs.
# The paper does not explicitly specify a batch size
# for the STS-B fine-tuning experiment in the text.
#
# We therefore choose a hardware-feasible implementation.

PAPER_MAX_LENGTH = None

MAX_LENGTH = 128

PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

EFFECTIVE_BATCH_SIZE = (
    PHYSICAL_BATCH_SIZE *
    GRADIENT_ACCUMULATION_STEPS
)

# Validation after every epoch
EVAL_BATCH_SIZE = 32

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = torch.cuda.is_available()

SEED = 42


print("Model:", MODEL_NAME)
print("Device:", DEVICE)

print("\nTraining:")
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

print("\nBatching:")
print("Physical batch size:", PHYSICAL_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)

print("\nMax length:", MAX_LENGTH)
print("Seed:", SEED)

Model: bert-base-uncased
Device: cuda

Training:
Epochs: 4
Learning rate: 2e-05

Batching:
Physical batch size: 4
Gradient accumulation: 4
Effective batch size: 16

Max length: 128
Seed: 42


In [3]:
# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

print("Seed set to:", SEED)


Seed set to: 42


In [4]:
# ============================================================
# 4. LOAD STS-B
# ============================================================

STSB_PATH = STS_DIR / "STSb"

print("STS-B path:")
print(STSB_PATH.resolve())

if not STSB_PATH.exists():
    raise FileNotFoundError(
        f"STS-B dataset not found:\n{STSB_PATH.resolve()}"
    )


stsb_dataset = load_from_disk(STSB_PATH)

print("\nDataset:")
print(stsb_dataset)

STS-B path:
F:\SEM-7\NLP\Project\data\STS\STSb

Dataset:
DatasetDict({
    train: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 1379
    })
})


In [5]:
# ============================================================
# 5. VERIFY STS-B SPLITS
# ============================================================

print("Splits:")
print(stsb_dataset.keys())

for split in stsb_dataset.keys():

    print(
        f"{split:12s}: "
        f"{len(stsb_dataset[split]):,} examples"
    )

    print(
        "Columns:",
        stsb_dataset[split].column_names
    )

    print()

Splits:
dict_keys(['train', 'validation', 'test'])
train       : 5,749 examples
Columns: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']

validation  : 1,500 examples
Columns: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']

test        : 1,379 examples
Columns: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']



In [6]:
# ============================================================
# 6. CONVERT STS-B SPLITS TO DATAFRAMES
# ============================================================

train_df = stsb_dataset["train"].to_pandas()
dev_df = stsb_dataset["validation"].to_pandas()
test_df = stsb_dataset["test"].to_pandas()


print("Train:")
print(train_df.shape)
print(train_df.columns.tolist())

print("\nDev:")
print(dev_df.shape)
print(dev_df.columns.tolist())

print("\nTest:")
print(test_df.shape)
print(test_df.columns.tolist())

Train:
(5749, 8)
['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']

Dev:
(1500, 8)
['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']

Test:
(1379, 8)
['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']


In [7]:
# ============================================================
# 7. INSPECT EXAMPLES
# ============================================================

print("Sentence 1:")
print(train_df.iloc[0]["sentence1"])

print("\nSentence 2:")
print(train_df.iloc[0]["sentence2"])

print("\nScore:")
print(train_df.iloc[0]["score"])

Sentence 1:
A plane is taking off.

Sentence 2:
An air plane is taking off.

Score:
5.0


In [8]:
# ============================================================
# 8. LOAD NLI-TRAINED SBERT
# ============================================================

print("NLI checkpoint:")
print(NLI_MODEL_PATH.resolve())

if not NLI_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"NLI checkpoint not found:\n"
        f"{NLI_MODEL_PATH.resolve()}"
    )

if not NLI_TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"NLI tokenizer not found:\n"
        f"{NLI_TOKENIZER_PATH.resolve()}"
    )


tokenizer = AutoTokenizer.from_pretrained(
    NLI_TOKENIZER_PATH
)

encoder = AutoModel.from_pretrained(
    MODEL_NAME
)


checkpoint = torch.load(
    NLI_MODEL_PATH,
    map_location="cpu"
)

print("\nCheckpoint loaded.")
print("Checkpoint keys:")
print(list(checkpoint.keys()))

NLI checkpoint:
F:\SEM-7\NLP\Project\notebooks\outputs\sbert_nli_base\model\sbert_nli_base.pt


W0924 05:40:37.519000 5648 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



Checkpoint loaded.
Checkpoint keys:
['model_state_dict', 'model_name', 'pooling', 'num_labels', 'max_length', 'seed', 'epochs', 'learning_rate', 'effective_batch_size', 'physical_batch_size', 'gradient_accumulation_steps']


In [9]:
# ============================================================
# 9. LOAD TRAINED BERT WEIGHTS
# ============================================================

model_state_dict = checkpoint["model_state_dict"]


bert_state_dict = {
    key[len("bert."):]: value
    for key, value in model_state_dict.items()
    if key.startswith("bert.")
}


print(
    "Number of BERT parameters:",
    len(bert_state_dict)
)


missing_keys, unexpected_keys = encoder.load_state_dict(
    bert_state_dict,
    strict=False
)


print("\nMissing keys:", len(missing_keys))
print("Unexpected keys:", len(unexpected_keys))

if missing_keys:
    print("\nMissing:")
    print(missing_keys)

if unexpected_keys:
    print("\nUnexpected:")
    print(unexpected_keys)

Number of BERT parameters: 199

Missing keys: 0
Unexpected keys: 0


In [10]:
# ============================================================
# 10. MEAN POOLING
# ============================================================

def mean_pooling(
    last_hidden_state,
    attention_mask
):
    """
    Mean-pool token representations while
    ignoring padding tokens.
    """

    mask = attention_mask.unsqueeze(-1).expand(
        last_hidden_state.size()
    ).float()

    summed = torch.sum(
        last_hidden_state * mask,
        dim=1
    )

    counts = torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )

    return summed / counts

In [11]:
# ============================================================
# 11. SBERT REGRESSION MODEL
# ============================================================

class SBERTSTSRegressor(nn.Module):

    def __init__(self, encoder):
        super().__init__()

        self.encoder = encoder

    def mean_pooling(
        self,
        last_hidden_state,
        attention_mask
    ):

        mask = attention_mask.unsqueeze(-1).expand(
            last_hidden_state.size()
        ).float()

        summed = torch.sum(
            last_hidden_state * mask,
            dim=1
        )

        counts = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        return summed / counts

    def encode(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        embeddings = self.mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

        return embeddings

    def forward(
        self,
        sentence1_inputs,
        sentence2_inputs
    ):

        u = self.encode(
            sentence1_inputs["input_ids"],
            sentence1_inputs["attention_mask"]
        )

        v = self.encode(
            sentence2_inputs["input_ids"],
            sentence2_inputs["attention_mask"]
        )

        similarity = torch.nn.functional.cosine_similarity(
            u,
            v,
            dim=1
        )

        return similarity

In [12]:
# ============================================================
# 12. STS-B DATASET
# ============================================================

class STSBDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe):
        self.sentence1 = dataframe["sentence1"].astype(str).tolist()
        self.sentence2 = dataframe["sentence2"].astype(str).tolist()
        self.scores = dataframe["score"].astype(float).tolist()

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):

        return {
            "sentence1": self.sentence1[idx],
            "sentence2": self.sentence2[idx],
            "score": self.scores[idx]
        }


train_dataset = STSBDataset(train_df)
dev_dataset = STSBDataset(dev_df)
test_dataset = STSBDataset(test_df)


print("Train:", len(train_dataset))
print("Dev:", len(dev_dataset))
print("Test:", len(test_dataset))

Train: 5749
Dev: 1500
Test: 1379


In [13]:
# ============================================================
# 13. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    sentences1 = [
        item["sentence1"]
        for item in batch
    ]

    sentences2 = [
        item["sentence2"]
        for item in batch
    ]

    scores = torch.tensor(
        [
            item["score"]
            for item in batch
        ],
        dtype=torch.float
    )

    inputs1 = tokenizer(
        sentences1,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    inputs2 = tokenizer(
        sentences2,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        "sentence1": inputs1,
        "sentence2": inputs2,
        "scores": scores
    }

In [14]:
# ============================================================
# 14. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=PHYSICAL_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("Train batches:", len(train_loader))
print("Dev batches:", len(dev_loader))
print("Test batches:", len(test_loader))

Train batches: 1438
Dev batches: 47
Test batches: 44


In [15]:
# ============================================================
# 15. CREATE MODEL
# ============================================================

model = SBERTSTSRegressor(
    encoder
)

model = model.to(DEVICE)


print("Model loaded on:", DEVICE)
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

Model loaded on: cuda
Trainable parameters: 109482240


In [16]:
# ============================================================
# 16. LOSS AND OPTIMIZER
# ============================================================

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)

Loss: MSELoss()
Optimizer: Adam
Learning rate: 2e-05


In [17]:
# ============================================================
# 17. STS-B SCORE NORMALIZATION
# ============================================================

def normalize_stsb_scores(scores):
    """
    Convert STS-B scores from [0, 5] to [0, 1].
    """
    return scores / 5.0

In [18]:
# ============================================================
# 18. TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    scaler,
    device,
    accumulation_steps
):
    
    model.train()

    total_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(
        dataloader,
        desc="Training",
        leave=False
    )

    for step, batch in enumerate(progress_bar):

        inputs1 = {
            key: value.to(device)
            for key, value in batch["sentence1"].items()
        }

        inputs2 = {
            key: value.to(device)
            for key, value in batch["sentence2"].items()
        }

        scores = normalize_stsb_scores(
            batch["scores"]
        ).to(device)

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            predictions = model(
                inputs1,
                inputs2
            )

            loss = criterion(
                predictions,
                scores
            )

            loss = loss / accumulation_steps

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        if USE_AMP:

            scaler.scale(loss).backward()

        else:

            loss.backward()


        # ----------------------------------------------------
        # Optimizer step
        # ----------------------------------------------------

        if (
            (step + 1) % accumulation_steps == 0
            or
            (step + 1) == len(dataloader)
        ):

            if USE_AMP:

                scaler.step(optimizer)
                scaler.update()

            else:

                optimizer.step()

            optimizer.zero_grad(set_to_none=True)


        actual_loss = loss.item() * accumulation_steps

        total_loss += actual_loss

        progress_bar.set_postfix(
            loss=f"{actual_loss:.4f}"
        )


    return total_loss / len(dataloader)

In [19]:
# ============================================================
# 19. VALIDATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate_stsb(
    model,
    dataloader,
    device
):

    model.eval()

    predictions = []
    gold_scores = []

    for batch in tqdm(
        dataloader,
        desc="Evaluating",
        leave=False
    ):

        inputs1 = {
            key: value.to(device)
            for key, value in batch["sentence1"].items()
        }

        inputs2 = {
            key: value.to(device)
            for key, value in batch["sentence2"].items()
        }

        scores = batch["scores"].numpy()

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            similarity = model(
                inputs1,
                inputs2
            )

        similarity = similarity.float().cpu().numpy()

        predictions.extend(
            similarity.tolist()
        )

        gold_scores.extend(
            (scores / 5.0).tolist()
        )


    predictions = np.array(predictions)
    gold_scores = np.array(gold_scores)

    rho = spearmanr(
        predictions,
        gold_scores
    ).statistic

    return rho * 100

In [20]:
# ============================================================
# 20. AMP SCALER
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print("AMP enabled:", USE_AMP)

AMP enabled: True


In [22]:
# ============================================================
# 21. TRAINING LOOP
# ============================================================

best_dev_score = -float("inf")
best_epoch = None

history = []


for epoch in range(EPOCHS):

    print(
        f"\n{'=' * 60}"
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
    )

    print(
        f"{'=' * 60}"
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=DEVICE,
        accumulation_steps=GRADIENT_ACCUMULATION_STEPS
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    dev_spearman = evaluate_stsb(
        model=model,
        dataloader=dev_loader,
        device=DEVICE
    )


    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "dev_spearman": dev_spearman
    })


    print(
        f"\nTrain loss: {train_loss:.6f}"
    )

    print(
        f"Dev Spearman: {dev_spearman:.4f}"
    )


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if dev_spearman > best_dev_score:

        best_dev_score = dev_spearman
        best_epoch = epoch + 1

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": MODEL_NAME,
                "pooling": "mean",
                "max_length": MAX_LENGTH,
                "learning_rate": LEARNING_RATE,
                "epochs": EPOCHS,
                "seed": SEED,
                "best_epoch": best_epoch,
                "best_dev_spearman": best_dev_score,
            },
            OUTPUT_DIR / "sbert_nli_stsb_best.pt"
        )

        print(
            f"Best model saved "
            f"(epoch {best_epoch})"
        )


print("\nTraining complete.")

print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best dev Spearman: {best_dev_score:.4f}"
)


Epoch 1/4



Train loss: 0.025912
Dev Spearman: 85.7409
Best model saved (epoch 1)

Epoch 2/4



Train loss: 0.010648
Dev Spearman: 86.4456
Best model saved (epoch 2)

Epoch 3/4



Train loss: 0.006363
Dev Spearman: 86.6046
Best model saved (epoch 3)

Epoch 4/4



Train loss: 0.004982
Dev Spearman: 86.7280
Best model saved (epoch 4)

Training complete.
Best epoch: 4
Best dev Spearman: 86.7280


In [23]:
# ============================================================
# 22. TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(history)

history_df

,epoch,train_loss,dev_spearman
0,1,0.025912,85.740949
1,2,0.010648,86.445574
2,3,0.006363,86.604568
3,4,0.004982,86.727968


In [25]:
# ============================================================
# 23. LOAD BEST MODEL
# ============================================================

BEST_MODEL_PATH = (
    OUTPUT_DIR /
    "sbert_nli_stsb_best.pt"
)

best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model = model.to(DEVICE)
model.eval()

print("Best model loaded.")

print(
    "Best epoch:",
    best_checkpoint["best_epoch"]
)

print(
    "Best dev Spearman:",
    best_checkpoint["best_dev_spearman"]
)

Best model loaded.
Best epoch: 4
Best dev Spearman: 86.72796768107763


In [26]:
# ============================================================
# 24. FINAL TEST EVALUATION
# ============================================================

test_spearman = evaluate_stsb(
    model=model,
    dataloader=test_loader,
    device=DEVICE
)

print(
    f"STS-B Test Spearman: "
    f"{test_spearman:.4f}"
)

STS-B Test Spearman: 84.3706


In [27]:
# ============================================================
# 25. TABLE 2 COMPARISON
# ============================================================

paper_score = 85.35

comparison = pd.DataFrame([
    {
        "Model": "SBERT-NLI-STSb-base",
        "Paper": paper_score,
        "Our Result": test_spearman,
        "Difference": test_spearman - paper_score
    }
])

comparison

,Model,Paper,Our Result,Difference
0,SBERT-NLI-STSb-base,85.35,84.370561,-0.979439
